In [ ]:
import pandas as pd
import google.generativeai as genai
import time
from sklearn.metrics import classification_report, accuracy_score

# --- 1. CẤU HÌNH API ---
try:
    with open("API.txt", "r") as f:
        GOOGLE_API_KEY = f.read().strip()
    genai.configure(api_key=GOOGLE_API_KEY)
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file API.txt!")
    exit()

# Cấu hình an toàn (Tắt bộ lọc để Gemini có thể phân tích câu chửi)
safety_settings = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

# System Prompt định nghĩa luật chơi
SYSTEM_PROMPT = """
Bạn là một chuyên gia kiểm duyệt nội dung. Nhiệm vụ của bạn là phân loại bình luận tiếng Việt thành 3 nhãn:
- 0 (CLEAN): Bình thường, không xúc phạm, hoặc chỉ là câu cảm thán nhẹ nhàng không tấn công ai.
- 1 (OFFENSIVE): Chứa từ tục tĩu, chửi thề, tiếng lóng thô tục nhưng KHÔNG nhắm vào đối tượng cụ thể.
- 2 (HATE): Lăng mạ, quấy rối, công kích trực tiếp cá nhân/nhóm/tổ chức hoặc phân biệt vùng miền/chủng tộc.

CHỈ TRẢ VỀ DUY NHẤT SỐ 0, 1 HOẶC 2. KHÔNG GIẢI THÍCH GÌ THÊM.
"""

model = genai.GenerativeModel(
    model_name='gemini-2.5-flash', # Dùng flash cho nhanh và tiết kiệm
    system_instruction=SYSTEM_PROMPT,
    safety_settings=safety_settings
)

# --- 2. ĐỌC DỮ LIỆU ---
df = pd.read_csv('test.csv')

# Danh sách lưu kết quả dự đoán của AI
y_true = df['label_id'].tolist()
y_pred = []

print(f"Bắt đầu đánh giá {len(df)} dòng dữ liệu...")

# --- 3. GỌI API ĐÁNH GIÁ ---
for index, row in df.iterrows():
    comment = row['free_text']
    
    try:
        # Gọi Gemini
        response = model.generate_content(f"Phân loại câu này: {comment}")
        
        # Lấy text trả về, xóa khoảng trắng và chuyển về int
        ans = response.text.strip()
        
        # Đề phòng Gemini trả về text dài, ta chỉ lấy số đầu tiên thấy được
        if '0' in ans: prediction = 0
        elif '1' in ans: prediction = 1
        elif '2' in ans: prediction = 2
        else: prediction = 0 # Mặc định nếu lỗi
            
        y_pred.append(prediction)
        print(f"Dòng {index}: Thực tế {row['label_id']} - Gemini dự đoán {prediction}")
        
        # Nghỉ 1 chút để tránh rate limit nếu dùng bản free
        time.sleep(1) 
        
    except Exception as e:
        print(f"Lỗi ở dòng {index}: {e}")
        y_pred.append(0) # Gán tạm nhãn 0 nếu lỗi API

# --- 4. XUẤT BÁO CÁO ---
print("\n" + "="*30)
print("KẾT QUẢ ĐÁNH GIÁ CỦA GEMINI")
print("="*30)

# In báo cáo chi tiết Precision, Recall, F1
print(classification_report(y_true, y_pred, target_names=["CLEAN (0)", "OFFENSIVE (1)", "HATE (2)"], labels=[0, 1, 2]))

# Lưu kết quả so sánh ra file mới để xem lại
df['gemini_pred'] = y_pred
df.to_csv('gemini_evaluation_results.csv', index=False, encoding='utf-8-sig')
print("\nĐã lưu chi tiết so sánh vào file: gemini_evaluation_results.csv")